In [ ]:
import os
from langchain.memory import ConversationBufferWindowMemory

from groq import Groq

memory = ConversationBufferWindowMemory( k=1)

client = Groq(
    api_key=,
)

In [123]:
all_terms = search_terms_generation("What are the reasons of inflation?")
print(all_terms)

What causes inflation
Why is inflation happening
What are the causes of inflation
Why is there inflation
What is inflation
What are the reasons for inflation
Why is inflation a problem
What is the meaning of inflation
What are the causes of economic inflation
Why is inflation a concern
What is the definition of inflation
What are the causes of economic instability
Why is inflation a major issue
What are the causes of inflation in the economy
Why is inflation a problem for consumers
What are the causes of inflation in the US
Why is inflation a concern for businesses
What are the causes of inflation in the world
Why is inflation a major economic issue


In [27]:
from langchain.embeddings import HuggingFaceEmbeddings

# Load Embedding Model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

def generate_embeddings(text):
    return embedding_model.embed_query(text)

C:\Users\Puroshotam S\AppData\Local\Temp\ipykernel_15524\3786475977.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
d:\Self\Knowledge-Hub-Assistant\open\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Self\Knowledge-Hub-Assistant\open\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your m

In [28]:
embedd = generate_embeddings("What are the reasons of inflation?")

In [ ]:

# memory.save_context({"input": "hi"}, {"output": "whats up"})
# memory.save_context({"input": "not much you"}, {"output": "not much"})

### Hashi Corp Vault

In [56]:
# Creating client

import hvac
client = hvac.Client(url='http://127.0.0.1:8200')

In [68]:
# Creating/updating secret

client.secrets.kv.v2.create_or_update_secret(
    path='hvac', 
    secret={'pssst': 'this is secret'}, 
    mount_point='secret'  # Use 'secret' as the mount point
)

{'request_id': '6db70820-16cc-c06d-4692-e22ab812d3cd',
 'lease_id': '',
 'renewable': False,
 'lease_duration': 0,
 'data': {'created_time': '2025-01-08T09:01:22.4276457Z',
  'custom_metadata': None,
  'deletion_time': '',
  'destroyed': False,
  'version': 1},
 'wrap_info': None,
 'warnings': None,
 'auth': None,
 'mount_type': 'kv'}

In [69]:
# Reading secret

mount_point = 'secret'
secret_path = 'hvac'

read_secret_result = client.secrets.kv.v2.read_secret(
    path=secret_path,
    mount_point=mount_point,
)
print(read_secret_result['data']['data'])

{'pssst': 'this is secret'}


In [61]:
# Deleting latest version of secret
client.secrets.kv.v2.delete_latest_version_of_secret(
    path='hvac',
)

<Response [204]>

In [65]:
# Deleting specific version of secret
client.secrets.kv.v2.delete_secret_versions(
    path='hvac',
    versions=[1, 2, 3],
)

<Response [204]>

In [66]:
# Destroying everything of that secret
client.secrets.kv.v2.delete_metadata_and_all_versions(
    path='hvac',
)

<Response [204]>

In [ ]:
from groq import Groq

client = Groq(
    api_key=,
)

def classify_and_respond(user_input):
    prompt = f"""
            You are a highly intelligent assistant designed to analyze user questions related to real estate and classify them into two categories:

            1. "dataset_query": These are questions requiring information from real estate data, such as prices, locations, or property details.
            2. "greeting" or "miscellaneous": These are general greetings or unrelated conversational inputs.

            Your task is to analyze the user's input and respond in one of the following ways:
            - If the input is classified as **dataset_query**, respond with exactly:  
            "dataset_query"  
            - If the input is classified as **greeting** or **miscellaneous**, provide a natural and appropriate conversational response.

            Strictly follow these rules and examples:
            Examples:
            Question: "What is the average price of apartments in New York?"
            Output: "dataset_query"

            Question: "Hello, how are you?"
            Output: "Hello! How can I assist you?"

            Question: "Can you tell me about 3-bedroom houses available in San Francisco?"
            Output: "dataset_query"

            Question: "Good morning!"
            Output: "Good morning! Let me know if you have any questions."

            When a user provides an input, analyze it carefully and provide either "dataset_query" or a conversational response, depending on the classification.

            Question: "{user_input}"
            Output:
            """
    
    chat_completion = client.chat.completions.create(
        messages=[{'role': 'system', 'content': 'You are a classifier.'},
                {"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
        temperature=0,
        max_tokens=1024,
    )

    # print(chat_completion.choices[0].message.content)

    return chat_completion.choices[0].message.content.strip().lower()


In [ ]:
import sqlite3
import pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

class CodeGeneratorAgent:
    def __init__(self, llm):
        self.llm = llm

    def generate_sql_query(self, query, db_info, sample_records):
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are an assistant for generating SQL queries for an SQLite database.
                    The database schema and details are provided below:
                    Table name: dataTable
                    Schema: {db_info}
                    Sample records: {sample_records}
                    
                    Always evaluate the user's question against the following criteria. If any criterion is met, respond with a one-liner follow-up asking for clarification:
                    1. The user's question is incomplete, irrelevant, empty or does not provide enough information for a meaningful response.
                    2. The user's question is not related to the given table and column schema.
                    3. Provide a direct answer to valid questions without asking unnecessary clarifications.
                    4. Ask for clarification only when the question is genuinely unclear or lacks essential details.
                    5. If user query is greeting or not relevant for SQL query then generate response with greeting and follow up question to define user criteria for real estate property selection or filtering.
                    
                    THE RESPONSE MUST BE STRICTLY ONLY THE SQL QUERY. DO NOT INCLUDE ANY TAGS LIKE ```sql``` OR ANY SORT OF EXPLANATIONS. JUST QUERY, AS IT WILL BE DIRECTLY USED IN SQL QUERY ENGINE.
                    """,
                ),
                ("human", "User Query: {query}"),
            ]
        )

        # Use the LLM to generate SQL query
        chain = prompt | self.llm
        response = chain.invoke({"db_info": db_info, 
                                 "sample_records": sample_records,
                                 "query": query})
        return response.content.strip()  # Remove extra whitespace or newlines

# Code Executor Agent (SQL Query Executor)
class CodeExecutorAgent:
    def __init__(self, db_connection):
        self.db_connection = db_connection

    def execute_sql_query(self, sql_query):
        try:
            cursor = self.db_connection.cursor()
            cursor.execute(sql_query)
            result = cursor.fetchall()  # Fetch all results from the query execution
            return result
        except Exception as e:
            return f"Error executing SQL query: {str(e)}"

# Insight Generator Agent
class InsightGeneratorAgent:
    def __init__(self, llm):
        self.llm = llm

    def generate_insight(self, user_query, sql_query, execution_result):
        prompt = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are a real estate assistant with ARABIC native language for converting the result into a natural language response to user's query.
                    The result is from executing an SQL query on an SQLite database, and you need to generate natural language response in arabic language from it.
                    user query: {user_query}
                    sql query: {sql_query}
                    Execution Result: {execution_result}
                    Provide the recommendations as natural language response in ARABIC LANGUAGE based on the execution result.
                    
                    for e.g.
                    user query: "Give me the count of records in dataTable?"
                    sql query: SELECT COUNT(*) FROM dataTable
                    Execution Result: [(200,)]
                    Response: "يحتوي الجدول على 200 نقطة بيانات."
                    """,
                ),
            ]
        )

        # Generate insight from execution result
        chain = prompt | self.llm
        response = chain.invoke({"user_query": user_query,
                                 "sql_query": sql_query,
                                 "execution_result": execution_result})
        return response.content.strip()

# Convert CSV to SQLite Database
def csv_to_sqlite(csv_file_path, sqlite_db_path):
    # Load CSV into pandas DataFrame
    df = pd.read_csv(csv_file_path, encoding='latin1')
    # df = pd.read_csv()
    # Create SQLite database and write the DataFrame to it
    conn = sqlite3.connect(sqlite_db_path)
    df.to_sql('dataTable', conn, if_exists='replace', index=False)
    sample_records = df.head(5).to_dict(orient="records")
    return conn, sample_records

# Main Logic
def main(user_query, csv_file_path, sqlite_db_path):
    # Step 1: Convert CSV to SQLite
    db_connection, sample_records = csv_to_sqlite(csv_file_path, sqlite_db_path)
    
    # Step 2: Get database schema (tables and column info)
    db_info = get_db_schema(db_connection)
    # print("database info: \n", db_info)
    # Initialize agents
    code_generator = CodeGeneratorAgent(llm=model)
    code_executor = CodeExecutorAgent(db_connection=db_connection)
    insight_generator = InsightGeneratorAgent(llm=model)

    classification = classify_and_respond(user_query)
    
    if "dataset_query" in classification:
        generated_sql_query = code_generator.generate_sql_query(query=user_query, db_info=db_info, sample_records=sample_records)
        print("Generated SQL Query:\n", generated_sql_query)
        if "SELECT" or "Select" in generated_sql_query:
            execution_result = code_executor.execute_sql_query(generated_sql_query)
            print("Execution Result:\n", execution_result)

            # Step 5: Generate insights from the execution result (Agent C)
            insight = insight_generator.generate_insight(user_query=user_query,sql_query=generated_sql_query, execution_result=execution_result)
            # print("Insight:\n", insight)

            return insight
        return generated_sql_query
    else:
        return classification

# Get database schema
def get_db_schema(db_connection):
    cursor = db_connection.cursor()
    cursor.execute("PRAGMA table_info(dataTable);")
    schema_info = cursor.fetchall()
    db_info = "\n".join([f"Column: {col[1]}, Type: {col[2]}" for col in schema_info])
    
    return db_info


model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key = 
    # other params...
)
if __name__ == "__main__":
    # user_query = "Recommend me 3 stocks which have average price closer to that of 'Armor Plate Stryker'"
    user_query = "Show me the top 3 properties"
    csv_file_path = r"Recommender - Sample Data(units_av).csv"  # Path to your CSV file
    sqlite_db_path = r"real_estate.db"   # Path to the SQLite database

    # Call the main function
    print("Query: ", user_query)
    insight = main(user_query, csv_file_path, sqlite_db_path)
    print("Final Insight:", insight)

Query:  Show me the top 3 properties
Generated SQL Query:
 SELECT * FROM dataTable ORDER BY price DESC LIMIT 3
Execution Result:
 [(1176, '????? - ???? ????????  - ?????? ????', None, None, 5, 4, 31, 4, '??????', None, 2, 20.0, 1, 1, 24.818111, 46.641573, '8/8/2023', '02-04-0054', 68.0, None, '02-04-0054-0000000002', 3350000.0, 3350000.0, '27', None, None, 420.0, 540.0, 1, 1.0, '???', 4, 7, 0, 0, None, '???', '1', None, 9.0, 46.9, 44.3, 34.1, 27.8, 3350000.0, 1, 1, 1, 1, 0, 1), (1176, '????? - ???? ????????  - ?????? ????', None, None, 5, 4, 31, 4, '??????', None, 2, 20.0, 1, 1, 24.818111, 46.641573, '8/8/2023', '02-04-0054', 68.0, None, '02-04-0054-0000000003', 3350000.0, 3350000.0, '27', None, None, 420.0, 540.0, 1, 1.0, '???', 4, 7, 0, 0, None, '???', '1', None, 9.0, 46.9, 44.3, 34.1, 27.8, 3350000.0, 1, 1, 1, 1, 0, 1), (1176, '????? - ???? ????????  - ?????? ????', None, None, 5, 4, 31, 4, '??????', None, 2, 20.0, 1, 1, 24.818111, 46.641573, '8/8/2023', '02-04-0054', 68.0, None, '0

In [ ]:
# def prompt_creation(user_query, history):
#     prompt = f"""You are a highly knowledgeable and professional real estate agent chatbot. Your primary responsibility is to assist users with property-related inquiries by providing clear, relevant, and professional responses. Leverage prior interactions to maintain continuity and coherence throughout the conversation.

#         ### Guidelines
#         1. Carefully analyze both the **conversation history** and the **current user query**.
#         2. Use context from the conversation history to avoid redundant information and offer smooth, follow-up answers.
#         3. Before recommending any properties, ensure that essential user details are gathered by checking the following checklist in the conversation history:
#             - The user's **name** to personalize future responses This must be the first question from User. If user is not comfortable in sharing his/her name then move on.
#             - Specific **property preferences**, including location, price range, number of bedrooms, and type of property.
#         If any of these details are missing, initiate a friendly dialogue to collect the necessary information **one question at a time**, rather than asking all follow-up questions at once.
#         4. Do not immediately answer property-specific questions without establishing a foundation of user preferences for a more tailored response.
#         5. Use collected details to enhance the relevance and personalization of answers.
#         6. For questions unrelated to real estate, respond courteously and guide users back to relevant topics.
#         7. NEVER ASK 2 OR MORE FOLLOW UP QUESTIONS IN SINGLE QUESTION. ALWAYS ASK SINGLE FOLLOW UP QUESTION.

#         Example:
#         Question: "Good morning!"
#         Output: "Good morning! How can I assist you with your property search today?"

#         Question: "Hi"
#         Output: "Hello! How can I help you get your desired properties or answer any real estate-related questions today?"

#         Question: "Hi, I would like to see some of the properties."
#         Output: "Certainly! Before we proceed, could you please share your name and a few details about your preferences, such as your desired location and budget? This will help me provide more personalized recommendations.\n Please tell me your name."

#         Conversation History: {history}

#         Question: "{user_query}"
#         Output:
#         """
    
#     return prompt

In [ ]:
from groq import Groq

client = Groq(
    api_key=,
)

def bot_response(prompt, language='English'):
    
    chat_completion = client.chat.completions.create(
        messages=[{'role': 'system', 'content': f'You are a real estate agent who talk in {language} language and helps customer in finding and buying properties in a very professional and polite way.'},
                {"role": "user", "content": prompt}],
        model="llama-3.2-90b-vision-preview",
        temperature=0,
        max_tokens=1024,
    )
    return chat_completion.choices[0].message.content.strip()

In [52]:
def prompt_creation(user_query, history):
    prompt = f"""You are a highly knowledgeable and professional real estate agent chatbot. Your primary responsibility is to assist users with property-related inquiries by providing clear, relevant, and professional responses. Leverage prior interactions to maintain continuity and coherence throughout the conversation.

        ### Guidelines
        1. Carefully analyze both the **conversation history** and the **current user query**.
        2. Use context from the conversation history to avoid redundant information and offer smooth, follow-up answers.
        3. Before recommending any properties, ensure that essential user details are gathered by checking the following checklist in the conversation history:
            - User's **name** to address user in future responses. THIS MUST BE THE FIRST QUESTION FOR USER ALWAYS. If user is not comfortable in sharing his/her name then move on.
            - Specific **property preferences** like (Ask below these in single question):
                - Location
                - Price range
                - Type of property
        If any of these details are missing, initiate a friendly dialogue to collect the missed information.
        Once User respond with all his/her property preferences then in response mention only "TERMINATE FLOW".
        4. Do not immediately answer property-specific questions without establishing a foundation of user preferences for a more tailored response.
        5. Use collected details to enhance the relevance and personalization of answers.
        6. For questions unrelated to real estate, respond courteously and guide users back to relevant topics.
        7. Always generate **very short**, crisp, precise, polite, generous and real estate professional response. Do not generate lengthy response.

        Example:
        Question: "Good morning!"
        Output: "Good morning! How can I assist you with your property search today?"

        Question: "Hi"
        Output: "Hello! How can I help you get your desired properties?"

        Question: "Hi, I would like to see some of the properties."
        Output: "Certainly! Before we proceed, may I have your name?"

        Conversation History: {history}

        Question: "{user_query}"
        Output:
        """
    
    return prompt

In [54]:
history = ""
while True:
    
    if not history:
        print("Agent: Hi, I am real estate AI assistant. May I have your name?")
        history += "Agent: Hi, I am real estate AI assistant. May I have your name?\n"

    user_query = input("You: ")
    if user_query.lower() in ["quit", "exit", "bye"]:
        print("\nEnding conversation...\n")
        break

    history += "User: " + user_query + '\n'
    res = bot_response(prompt_creation(user_query, history), language='Arabic')
    history += "Agent: " + res + '\n'
    
    print("Response: ", res)
    print("\nHistory: ", history)

Agent: Hi, I am real estate AI assistant. May I have your name?
Response:  مرحباً أليكس، كيف يمكنني مساعدتك في البحث عن العقارات اليوم؟ هل يمكنك إخبارني عن تفضيلاتك في العقارات مثل الموقع، ومدى السعر، ونوع العقار؟

History:  Agent: Hi, I am real estate AI assistant. May I have your name?
User: Alex
Agent: مرحباً أليكس، كيف يمكنني مساعدتك في البحث عن العقارات اليوم؟ هل يمكنك إخبارني عن تفضيلاتك في العقارات مثل الموقع، ومدى السعر، ونوع العقار؟


Ending conversation...



In [2]:
def csv_to_sqlite(csv_file_path, sqlite_db_path):
    # Load CSV into pandas DataFrame
    df = pd.read_csv(csv_file_path, encoding='utf-8')
    # df = pd.read_csv()
    # Create SQLite database and write the DataFrame to it
    conn = sqlite3.connect(sqlite_db_path)
    df.to_sql('real_estate', conn, if_exists='replace', index=False)

In [2]:
import pandas as pd

In [4]:
df = pd.read_csv('df_english_v2.csv')
df.head()

,Apartment_code,project_id,Project URL,project_name_eng,city_id_eng,region_id_eng,District_Residential_area_eng,allow_payment_by_cash,allow_payment_by_loan,project_latitude,...,bathroom_count,apartment_type_eng,number_of_rooms,master_bedroom_size,living_room_size,kitchen_size,guestroom_size,apartment_for_sakani_beneficiary,apartment_for_non_sakani_beneficiary,construction_status_eng
0,A00001,77,https://sakani.sa/app/offplan-projects/77,Tabuk - Modern houses - Dora Tabuk,Tabuk,Tabuk,Al-Musayyaf,1,1,28.455361,...,4,apartment,3.0,18.7,16.0,8.8,23.0,0,0,Not started - selling on the map
1,A00002,77,https://sakani.sa/app/offplan-projects/77,Tabuk - Modern houses - Dora Tabuk,Tabuk,Tabuk,Al-Musayyaf,1,1,28.455361,...,4,apartment,3.0,18.7,13.6,8.8,15.6,0,0,Not started - selling on the map
2,A00003,77,https://sakani.sa/app/offplan-projects/77,Tabuk - Modern houses - Dora Tabuk,Tabuk,Tabuk,Al-Musayyaf,1,1,28.455361,...,4,apartment,3.0,18.7,13.6,8.8,15.6,0,0,Not started - selling on the map
3,A00004,77,https://sakani.sa/app/offplan-projects/77,Tabuk - Modern houses - Dora Tabuk,Tabuk,Tabuk,Al-Musayyaf,1,1,28.455361,...,4,apartment,3.0,18.7,13.6,8.8,15.6,0,0,Not started - selling on the map
4,A00005,77,https://sakani.sa/app/offplan-projects/77,Tabuk - Modern houses - Dora Tabuk,Tabuk,Tabuk,Al-Musayyaf,1,1,28.455361,...,4,apartment,3.0,18.7,13.6,8.8,15.6,0,0,Not started - selling on the map


In [13]:
import numpy as np

# Define aggregation functions for each column
agg_dict = {
    'Apartment_code': 'size',
    # 'Project URL': 'first',  # Assuming one URL per project
    # 'project_name_eng': 'first',  # Assuming one name per project
    # 'city_id_eng': 'first',  # Assuming one city per project
    # 'region_id_eng': 'first',  # Assuming one region per project
    # 'District_Residential_area_eng': 'first',  # Assuming one district per project
    'allow_payment_by_cash': 'max',  # Assuming it's binary (0 or 1), take max
    'allow_payment_by_loan': 'max',  # Assuming it's binary (0 or 1), take max
    # 'project_latitude': 'mean',  # Assuming averaging latitudes is meaningful
    # 'project_longitude': 'mean',  # Assuming averaging longitudes is meaningful
    'publish_date': 'first',  # Assuming one publish date per project
    'Construction_Completion_Percentage': 'max',
    'sakani_beneficiary_price': ['min', 'max'],
    'non_sakani_beneficiary_price': ['min', 'max'],
    'apartment_area_meter': ['min', 'max'],
    'living_area': ['min', 'max', 'mean'],
    'floor': lambda x: list(np.unique(x)),  # Unique values as a list
    'bathroom_count': ['min', 'max'],  # Aggregating numerical values
    'apartment_type_eng': lambda x: list(np.unique(x)),  # Unique apartment types
    'number_of_rooms': ['min', 'max'],  # Room counts range
    'master_bedroom_size': ['min', 'max'],  # Size range
    'living_room_size': ['min', 'max'],  # Size range
    'kitchen_size': ['min', 'max'],  # Size range
    'guestroom_size': ['min', 'max'],  # Size range
    # 'apartment_for_sakani_beneficiary': 'sum',  # Count beneficiaries
    # 'apartment_for_non_sakani_beneficiary': 'sum',  # Count non-beneficiaries
    # 'construction_status_eng': 'first',  # Unique statuses
}

# Apply the groupby and aggregation
result = df.groupby(
    by=['project_id', 'Project URL', 'project_name_eng', 'city_id_eng', 
        'region_id_eng', 'District_Residential_area_eng', 
        'project_latitude', 'project_longitude', 'publish_date']
).agg(agg_dict)

# Flatten multi-level column index if needed
result.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in result.columns]

# Reset the index if necessary
result = result.reset_index()

# Display the result
result.head()

,project_id,Project URL,project_name_eng,city_id_eng,region_id_eng,District_Residential_area_eng,project_latitude,project_longitude,publish_date,Apartment_code_size,...,number_of_rooms_min,number_of_rooms_max,master_bedroom_size_min,master_bedroom_size_max,living_room_size_min,living_room_size_max,kitchen_size_min,kitchen_size_max,guestroom_size_min,guestroom_size_max
0,8,https://sakani.sa/app/offplan-projects/8,Taif - Supplies - Al -Fateh,Taif,Jeddah,As-Sail Al-Saghir,21.524833,40.508167,8/21/2023,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9,https://sakani.sa/app/offplan-projects/9,Taif - Administration for Development and Deve...,Taif,Jeddah,Al-Dabt,21.377583,40.429222,2/25/2020,58,...,4.0,7.0,28.1,28.1,39.0,39.0,18.6,18.6,28.0,28.0
2,11,https://sakani.sa/app/offplan-projects/11,Makkah Al -Mukarramah - Five - Retaj,Mecca,Jeddah,Al-Nuwariya,21.556583,39.798583,1/23/2022,17,...,7.0,7.0,23.0,23.0,18.0,18.0,15.0,15.0,21.0,21.0
3,15,https://sakani.sa/app/offplan-projects/15,Abha - Ali Shar - Abha Hills,Abha,Asir,Al-Ta'awun,18.286109,42.513347,1/23/2023,70,...,4.0,5.0,22.5,33.0,36.0,36.0,9.5,16.4,25.0,40.0
4,16,https://sakani.sa/app/offplan-projects/16,Khamis Mushait - Abdul Rahman Al -Rashed - Al ...,Khamis Mushait,Asir,Al-Raqi,18.357650,42.753309,1/25/2021,277,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
result.shape

(127, 37)

In [25]:
for col in result.columns:
    if result[col].apply(lambda x: isinstance(x, list)).any():
        result[col] = result[col].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else x)
        

In [28]:
result=result.rename({"floor_<lambda>":"floor","apartment_type_eng_<lambda>":"apartment_type_eng"},axis="columns")
result[['floor']]

,floor
0,"0, 1, 2"
1,"0, 1"
2,"0, 1, 3"
3,0
4,0
...,...
122,"1, 2"
123,"2, 3, 4"
124,"2, 3"
125,"0, 1, 2, 3, 5"


In [30]:
import sqlite3

conn = sqlite3.connect('real_estate.db')
result.to_sql('project_details', conn, if_exists='replace', index=False)

127

In [31]:
unit_df = df[["Apartment_code", "project_id", "apartment_area_meter", "living_area", "floor", "bathroom_count", "apartment_type_eng", "number_of_rooms", "master_bedroom_size", "living_room_size", "kitchen_size", "guestroom_size"]]
unit_df.head()

,Apartment_code,project_id,apartment_area_meter,living_area,floor,bathroom_count,apartment_type_eng,number_of_rooms,master_bedroom_size,living_room_size,kitchen_size,guestroom_size
0,A00001,77,152.5,19.76,0,4,apartment,3.0,18.7,16.0,8.8,23.0
1,A00002,77,174.0,21.92,3,4,apartment,3.0,18.7,13.6,8.8,15.6
2,A00003,77,174.0,21.92,4,4,apartment,3.0,18.7,13.6,8.8,15.6
3,A00004,77,174.0,21.92,5,4,apartment,3.0,18.7,13.6,8.8,15.6
4,A00005,77,174.0,21.92,6,4,apartment,3.0,18.7,13.6,8.8,15.6


In [32]:
import sqlite3

conn = sqlite3.connect('real_estate.db')
unit_df.to_sql('unit_details', conn, if_exists='replace', index=False)

5000

In [ ]:
client = Groq(api_key= os.getenv("GROQ_API_KEY"))

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key="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"
    # other params...
)


In [ ]:
prompt=f"""You are a Real Estate helpful assistant who helps user by recommending best properties based on user property preferences. Your task is to write the SQL query. 
    
    Generate a SQL query to fetch property recommendations from the database based on user preferences for location, property type, and budget. Consider the following constraints for dynamic recommendations:

    1. Fixed Location with Improved Property Features:
    Suggest properties that belong to the same location but offer better features or property types within the user's specified budget.

    2. Fixed Location with Lower Budget Alternatives:
    Suggest properties within the same location, having the same property type but at a lower budget than the user’s specified budget.

    Ensure the query selects only the relevant columns needed to generate clear, concise property recommendations rather than using SELECT *. Use appropriate ORDER BY clauses to rank better properties by area, amenities, or price-to-value ratio. Avoid unnecessary joins and ensure the query is optimized for performance.
    
    # Below is the user conversational history:
    {str(history)}

    response:
    """
chat_completion = client.chat.completions.create(
    messages=[{'role': 'system', 'content': "You are a Real Estate helpful assistant"},
            {"role": "user", "content": prompt}],
    model="llama-3.2-90b-vision-preview",
    temperature=0,
    max_tokens=1024,
)

print(chat_completion.choices[0].message.content.strip())

In [33]:
import json

In [64]:
class DataDictionaryPrompt():

    def __init__(self,) -> None:
        self.sqllite_db_path=r"D:\30. Open Source llm -RAG\PubSec-Info-Assistant-Offshore\backend\real_estate.db"
       
    def __get_data_dict(self):
        try:
            # Load the Excel sheet into a pandas DataFrame
            df = pd.read_excel(r"D:\30. Open Source llm -RAG\PubSec-Info-Assistant-Offshore\backend\Data_Dictionary_v3.xlsx",sheet_name="dict2")
            # print(df)
            # Create an in-memory SQLite database
            conn = sqlite3.connect(self.sqllite_db_path)

            # Load the DataFrame into the SQLite database
            df.to_sql("dict_data", conn, index=False, if_exists="replace")
            
            query="select * from dict_data"

            # Execute the SQL query
            df1 = pd.read_sql_query(query, conn)
            data_dict=df1.to_dict(orient="records")
            return data_dict  # Return the DataFrame with query results
            
        except Exception as e:
            print("SQL query didn't work due to:", e)
            return None
        finally:
            # Close the database connection
            conn.close()

    def __get_top3(self,table_name):
        try:
            # Load an SQLite database
            conn = sqlite3.connect(self.sqllite_db_path)
            
            query=f"""select * from {table_name} Limit 3"""

            # Execute the SQL query
            top_df = pd.read_sql_query(query, conn)

            return top_df  # Return the DataFrame with query results

        except Exception as e:
            print("Data Dictionary Prompt :: SQL query didn't work due to:", e)
            return None
        finally:
            # Close the database connection
            conn.close()

    def __get_table_details_with_columns(self):

        column_details=self.__get_data_dict()
        
        # Initialize a structure to hold the combined result
        table_details={1:{"table_name":"project_details","table_desc":"""The `Project_Details` table provides project-level information, including project identifiers, location details (city, region, latitude, longitude), construction status, pricing information for Sakani and non-Sakani beneficiaries, and available payment options. It also captures minimum and maximum ranges for unit dimensions and features within the project.""" },
        2:{"table_name":"unit_details","table_desc":"""The Unit_Details table contains information about individual apartment units within real estate projects, including attributes such as unique apartment identifiers, project associations, physical dimensions (e.g., area, room sizes), and specifications like floor number, number of rooms, and apartment type."""}
        }
        result = []
        table_ids=[1,2]
        # print(column_details)
        for table_id in table_ids:
            # print(table_id)
            table_name= table_details[table_id]["table_name"]
            table_desc=table_details[table_id]["table_desc"]

            print("Table Name ::", table_name,"ID ::",table_id)

            top_3=self.__get_top3(table_name)

            # Append the table details and its columns to the result
            result.append({
                "table_id": table_id,
                "table_name":table_name,
                "table_desc": table_desc,
                "top-3":top_3.to_csv(index=False),
                "columns": [
                    {
                        "col_id": column["column_id"],
                        "col_name": column["column_name"],
                        "col_type": column['dtypes'],
                        "col_desc": column["column_desc"]
                    }
                    for column in column_details if column["table_id"]==table_id
                ]
            })
        
        return json.dumps(result)

    def get_prompt(self):
        data_dictionary=self.__get_table_details_with_columns()
        data_dictionary_prompt = ''
        for table in json.loads(data_dictionary):
            data_dictionary_prompt += f"Table Name:{table['table_name']}\nTable Description:{table['table_desc']}"
            data_dictionary_prompt += "\nColumns(with data type and description):\n"
            for column in table['columns']:
                data_dictionary_prompt += f"{column['col_name']} ({column['col_type']}) : {column['col_desc']}\n"
            data_dictionary_prompt += f"""\n/* \n3 rows from {table['table_name']} table:\n"""
            data_dictionary_prompt+=table['top-3']
            data_dictionary_prompt += "*/ \n\n"
        return data_dictionary_prompt

In [65]:
dataobj=DataDictionaryPrompt()
dict_prompt=dataobj.get_prompt()

Table Name :: project_details ID :: 1
Table Name :: unit_details ID :: 2


In [67]:
# print(dict_prompt)